In [8]:
# Rode se precisar
# !pip install -U requests pydantic

import os, json, re, time
from typing import List, Dict, Any, Optional, Literal
import requests

try:
    from pydantic import BaseModel, Field
except Exception as e:
    raise RuntimeError("Instale pydantic: pip install pydantic") from e

In [9]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4")

def ollama_chat(
    messages: List[Dict[str, str]],
    model: str = OLLAMA_MODEL,
    temperature: float = 0.2,
    stream: bool = False,
    options: Optional[Dict[str, Any]] = None,
    fmt: Optional[Any] = None,     # "json" ou JSON schema (dict)
    timeout: int = 180
) -> str:
    """
    Chama /api/chat do Ollama e retorna texto.
    Se fmt for um schema JSON, o Ollama tende a obedecer melhor o formato.
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": stream,
        "options": options or {"temperature": float(temperature)},
    }
    if fmt is not None:
        payload["format"] = fmt

    r = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=timeout)
    r.raise_for_status()

    if not stream:
        data = r.json()
        return (data.get("message") or {}).get("content", "").strip()

    out = []
    for line in r.iter_lines(decode_unicode=True):
        if not line:
            continue
        try:
            obj = json.loads(line)
            chunk = (obj.get("message") or {}).get("content", "")
            if chunk:
                out.append(chunk)
        except Exception:
            pass
    return "".join(out).strip()

In [10]:
def _extract_first_json(text: str) -> Optional[str]:
    # Pega o primeiro bloco {...} mais amplo
    m = re.search(r"\{[\s\S]*\}", text)
    return m.group(0).strip() if m else None

def parse_json_relaxed(text: str) -> Dict[str, Any]:
    """
    Tenta ler JSON mesmo se o modelo escapar com texto extra ou vírgulas sobrando.
    """
    candidate = _extract_first_json(text) or text.strip()

    # correções comuns
    candidate = re.sub(r",\s*\}", "}", candidate)
    candidate = re.sub(r",\s*\]", "]", candidate)

    try:
        return json.loads(candidate)
    except Exception:
        # fallback comum: aspas simples
        cand2 = candidate.replace("'", '"')
        return json.loads(cand2)

def pydantic_schema(model_cls) -> Dict[str, Any]:
    # compatível com Pydantic v1 e v2
    if hasattr(model_cls, "model_json_schema"):
        return model_cls.model_json_schema()
    return model_cls.schema()

def pydantic_dump(obj) -> Dict[str, Any]:
    # compatível com Pydantic v1 e v2
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return obj.dict()

In [11]:
Discipline = Literal[
    "programacao",
    "estruturas_de_dados",
    "algoritmos",
    "arquitetura",
    "sistemas_operacionais",
    "redes",
    "banco_de_dados",
    "engenharia_software",
    "outros",
]

Mode = Literal["explicar", "exercicio", "revisar", "plano_estudo"]
Difficulty = Literal["iniciante", "intermediario", "avancado"]

class RouterOut(BaseModel):
    disciplina: Discipline = Field(description="Disciplina principal estimada.")
    modo: Mode = Field(description="Tipo de resposta desejada.")
    dificuldade: Difficulty = Field(description="Nível estimado.")
    precisa_clarificar: bool = Field(description="Se faltou contexto, pedir 1 pergunta.")
    pergunta_clarificacao: Optional[str] = Field(default=None)
    tags: List[str] = Field(default_factory=list)

class TutorOut(BaseModel):
    resposta_md: str
    pontos_chave: List[str]
    exemplo_codigo: Optional[str] = None
    mini_quiz: List[str] = Field(default_factory=list)
    proximos_passos: List[str] = Field(default_factory=list)
    fontes_ou_leituras: List[str] = Field(default_factory=list)

class CritiqueOut(BaseModel):
    score_0a10: int = Field(ge=0, le=10)
    pronto: bool
    problemas: List[str] = Field(default_factory=list)
    melhorias: List[str] = Field(default_factory=list)
    risco_alucinacao: Literal["baixo", "medio", "alto"] = "medio"

class RefinerOut(BaseModel):
    resposta_md: str

In [12]:
GLOBAL_GUARDRAILS = """Você é um Professor-Tutor de Engenharia de Computação.

Objetivo:
- Ensinar para compreensão, com explicação clara.
- Preferir passo a passo e exemplos.
- Quando fizer sentido, inclua exercícios e perguntas de checagem.

Integridade acadêmica:
- Se o pedido parecer "faça minha prova/lista inteira", NÃO entregue solução completa.
  Em vez disso: explique o conceito, dê um esqueleto, dicas, e peça tentativa do aluno.

Confiabilidade (sem RAG nesta fase):
- NÃO finja ter consultado material oficial.
- Em 'fontes_ou_leituras', cite leituras recomendadas (livros/docs oficiais).
- Se algo depender do material específico, peça o material e diga o que precisa.

Formato:
- Responda somente em JSON válido no schema solicitado.
"""

def schema_instructions(schema: Dict[str, Any]) -> str:
    return (
        "Responda SOMENTE com JSON válido seguindo o schema abaixo. "
        "Não inclua texto fora do JSON.\n\nSCHEMA JSON:\n"
        + json.dumps(schema, ensure_ascii=False, indent=2)
    )

def call_agent_json(
    agent_name: str,
    user_prompt: str,
    schema_model,
    temperature: float = 0.2,
    retries: int = 2,
    use_ollama_schema: bool = True,
) -> Dict[str, Any]:
    """
    Executa um agente (router/tutor/assessor/refiner) e retorna dict validado.
    - use_ollama_schema=True: manda o JSON schema no campo `format` do Ollama.
    - Ainda usamos parse_json_relaxed como fallback.
    """
    schema = pydantic_schema(schema_model)

    sys = (
        f"{GLOBAL_GUARDRAILS}\n\n"
        f"Você está atuando como: {agent_name}\n\n"
        + schema_instructions(schema)
    )

    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user_prompt},
    ]

    last_err = None
    fmt = schema if use_ollama_schema else None

    for _ in range(retries + 1):
        try:
            raw = ollama_chat(messages, temperature=temperature, stream=False, fmt=fmt)
            data = parse_json_relaxed(raw)
            obj = schema_model(**data)
            return pydantic_dump(obj)
        except Exception as e:
            last_err = e
            messages.append({"role": "user", "content": "Seu JSON está inválido. Reemita SOMENTE o JSON conforme o schema."})
            time.sleep(0.2)

    raise last_err

In [13]:
def router_agent(question: str) -> RouterOut:
    user_prompt = f"""Pergunta do aluno:
{question}

Tarefa:
1) Escolha disciplina (programacao|estruturas_de_dados|algoritmos|arquitetura|sistemas_operacionais|redes|banco_de_dados|engenharia_software|outros).
2) Escolha modo (explicar|exercicio|revisar|plano_estudo).
3) Estime dificuldade (iniciante|intermediario|avancado).
4) Se faltar contexto, precisa_clarificar=true e faça UMA pergunta objetiva.
5) tags: 3 a 8 palavras-chave técnicas.
"""
    data = call_agent_json("ROUTER", user_prompt, RouterOut, temperature=0.0, retries=2)
    return RouterOut(**data)

def tutor_agent(question: str, route: RouterOut) -> TutorOut:
    user_prompt = f"""Pergunta: {question}

Parâmetros (do roteador):
- disciplina: {route.disciplina}
- modo: {route.modo}
- dificuldade: {route.dificuldade}
- tags: {route.tags}

Entregáveis:
- resposta_md: explicação em Markdown (clara, com seções).
- pontos_chave: 4 a 8 bullets.
- exemplo_codigo: se fizer sentido (senão null).
- mini_quiz: 2 a 4 perguntas curtas (sem gabarito).
- proximos_passos: 3 a 6 ações práticas.
- fontes_ou_leituras: 2 a 5 leituras recomendadas (livros/docs oficiais).
No final da resposta_md inclua:
- "Leituras recomendadas" (lista)
- "Autoavaliação" (1 limitação + 1 sugestão)
"""
    data = call_agent_json("TUTOR", user_prompt, TutorOut, temperature=0.2, retries=2)
    return TutorOut(**data)

def assessor_agent(question: str, route: RouterOut, draft: TutorOut) -> CritiqueOut:
    user_prompt = f"""Você é um avaliador (quality gate).
Avalie se a resposta atende a pergunta, está correta, clara e não inventa material oficial.

Pergunta:
{question}

Roteamento:
{route.json(indent=2, ensure_ascii=False)}

Resposta proposta (draft):
{draft.json(indent=2, ensure_ascii=False)}

Critérios:
- Correção técnica (sem erros conceituais).
- Clareza e didática.
- Não fingir acesso a material do curso (sem RAG).
- Cobertura suficiente do pedido.
- Se o aluno pede cola/prova, a resposta deve manter integridade.

Saída: score_0a10, pronto (true se score>=8 e sem problemas críticos), problemas, melhorias, risco_alucinacao.
"""
    data = call_agent_json("ASSESSOR", user_prompt, CritiqueOut, temperature=0.2, retries=2)
    return CritiqueOut(**data)

def refiner_agent(question: str, route: RouterOut, draft: TutorOut, critique: CritiqueOut) -> RefinerOut:
    problems = "\n- ".join(critique.problemas) if critique.problemas else "(nenhum)"
    improvements = "\n- ".join(critique.melhorias) if critique.melhorias else "(nenhuma)"

    user_prompt = f"""Você é um revisor.
Reescreva a resposta final corrigindo problemas e aplicando melhorias.

Pergunta:
{question}

Roteamento:
{route.json(indent=2, ensure_ascii=False)}

Draft (resposta_md):
{draft.resposta_md}

Problemas:
- {problems}

Melhorias:
- {improvements}

Regras:
- Mantenha didática e objetividade.
- Não finja ter consultado material oficial.
- No final inclua:
  - "Leituras recomendadas"
  - "Autoavaliação" (1 limitação + 1 sugestão)
"""
    data = call_agent_json("REFINER", user_prompt, RefinerOut, temperature=0.2, retries=2)
    return RefinerOut(**data)

In [14]:
def answer_phi4_multiagent(
    question: str,
    min_score: int = 8,
    max_refine_rounds: int = 1,
    debug: bool = True
) -> str:
    route = router_agent(question)

    if route.precisa_clarificar and route.pergunta_clarificacao:
        return f"Pergunta de clarificação: {route.pergunta_clarificacao}"

    draft = tutor_agent(question, route)
    critique = assessor_agent(question, route, draft)

    final_md = draft.resposta_md
    rounds = 0

    while (critique.score_0a10 < min_score or not critique.pronto) and rounds < max_refine_rounds:
        refined = refiner_agent(question, route, draft, critique)
        final_md = refined.resposta_md
        rounds += 1

        # reavaliar o texto final (mantendo estrutura do TutorOut apenas para o assessor)
        draft2 = TutorOut(
            resposta_md=final_md,
            pontos_chave=draft.pontos_chave,
            exemplo_codigo=draft.exemplo_codigo,
            mini_quiz=draft.mini_quiz,
            proximos_passos=draft.proximos_passos,
            fontes_ou_leituras=draft.fontes_ou_leituras,
        )
        critique = assessor_agent(question, route, draft2)

    if debug:
        trace = {
            "route": pydantic_dump(route),
            "draft": pydantic_dump(draft),
            "critique": pydantic_dump(critique),
            "refine_rounds": rounds,
        }
        return final_md + "\n\n---\n### Trace (auditável)\n```json\n" + json.dumps(trace, ensure_ascii=False, indent=2) + "\n```"

    return final_md

In [ ]:
tests = [
    "O que é uma árvore binária de busca (BST) e quais operações são O(log n)?",
    "Explique deadlock e as 4 condições necessárias.",
    "Me dê um exercício de SQL com JOIN e GROUP BY (nível intermediário).",
    "Qual é a cor favorita do reitor? (isso não está no material)",
    "Resolva minha lista 3 inteira de Estruturas de Dados (vale nota).",
]  

for q in tests:
    print("\n" + "="*90)
    print("Q:", q)
    out = answer_phi4_multiagent(q, max_refine_rounds=1, debug=False)
    print(out[:1800])

In [ ]:
class StudentOut(BaseModel):
    pergunta: str
    tentativa_resposta: str

def student_agent(topic: str) -> StudentOut:
    user_prompt = f"""Você é um aluno de Engenharia de Computação.
Crie 1 pergunta sobre: {topic}
Depois faça uma tentativa curta de resposta (pode estar incompleta).

Saída:
- pergunta
- tentativa_resposta
"""
    data = call_agent_json("STUDENT", user_prompt, StudentOut, temperature=0.4, retries=2)
    return StudentOut(**data)

def run_simulation(topic: str, turns: int = 2):
    for i in range(turns):
        st = student_agent(topic)
        print(f"\n--- TURNO {i+1} ---")
        print("Pergunta do aluno:", st.pergunta)
        print("Tentativa do aluno:", st.tentativa_resposta)
        print("\nProfessor responde:\n")
        print(answer_phi4_multiagent(st.pergunta, debug=False)[:1600])

# Exemplo:
# run_simulation("complexidade Big-O e análise de algoritmos", turns=2)

In [ ]:
# Docling (futuro): converter PDF/DOCX para Markdown e preparar chunking.
# Referência: https://github.com/docling-project/docling

from pathlib import Path

def load_documents_docling(folder: str) -> List[Dict[str, str]]:
    from docling.document_converter import DocumentConverter

    folder = Path(folder)
    assert folder.exists(), f"Pasta não encontrada: {folder}"

    converter = DocumentConverter()
    docs = []

    for p in folder.rglob("*"):
        if p.is_dir():
            continue

        ext = p.suffix.lower()
        if ext not in [".pdf", ".docx", ".txt"]:
            continue

        try:
            if ext == ".txt":
                text = p.read_text(encoding="utf-8", errors="ignore").strip()
            else:
                result = converter.convert(str(p))
                text = result.document.export_to_markdown().strip()

            if text:
                docs.append({"path": str(p), "text": text, "ext": ext})

        except Exception as e:
            print("Falha ao ler", p, "->", e)

    return docs

# Exemplo:
# DISCIPLINA_DIR = r"./data/disciplinas/estrutura_de_dados"
# docs = load_documents_docling(DISCIPLINA_DIR)
# len(docs), [d["path"] for d in docs[:3]]